# Feature Engineering

Цель этого ноутбука - аккуратно выбрать две итоговые схемы признаков: одну для линейных моделей, вторую для деревьев. Отбор делаем по одному правилу: сначала максимизируем ROC-AUC, затем recall, затем precision.

Важный принцип: ноутбук нужен для исследования и объяснения выбора, а финальная сборка датасетов вынесена в DVC-команду. Это помогает не смешивать ручной анализ и воспроизводимый пайплайн.

## Что проверяем

Есть два слоя feature engineering:

1. Манипуляции с исходными признаками: оставить оригинал, заменить логарифмом или заменить набором dummy-признаков после дискретизации.
2. Маркеры риска из EDA: бинарные признаки и комбинации условий, которые показывали повышенную долю `target = 1`.

Для линейных моделей логарифмы и дискретизация особенно важны, потому что они помогают передать нелинейность. Для деревьев такие преобразования обычно менее критичны, поэтому отдельно проверяем, действительно ли они нужны.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from credit_risk_scoring.preprocessing.feature_engineering import (
    ALWAYS_FEATURE_COLUMNS,
    MARKER_COLUMNS,
    SELECTED_LINEAR_MARKERS,
    SELECTED_LINEAR_TRANSFORM,
    SELECTED_TREE_MARKERS,
    SELECTED_TREE_TRANSFORM,
    build_feature_space,
    evaluate_marker_combinations,
    evaluate_transform_combinations,
    make_marker_combinations,
    make_transform_combinations,
    sample_combinations,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 220)

In [ ]:
data = pd.read_csv(PROJECT_ROOT / "data" / "interim" / "train_prepared.csv")
data.shape

## Общее пространство признаков

Сначала создаём все кандидаты: служебные флаги, логарифмы, dummy-признаки после дискретизации и все маркеры. Дальше модели получают не всё сразу, а только выбранные комбинации колонок.

Служебные признаки всегда остаются в наборе: флаги пропусков из общей подготовки, флаги экстремальных значений `revolving_utilization` и `debt_ratio`, а также флаги системных кодов `96/98` для просрочек.

In [ ]:
feature_data, discretization_columns_new_names = build_feature_space(data)

print(f"rows: {feature_data.shape[0]:,}")
print(f"candidate columns: {feature_data.shape[1]:,}")
print(f"always-on features: {len(ALWAYS_FEATURE_COLUMNS)}")
print(f"discretized dummy columns: {sum(len(v) for v in discretization_columns_new_names.values())}")

## 1. Манипуляции с исходными признаками

Для каждого исходного признака выбираем ровно одно представление: оригинал, логарифм или дискретизацию. Для `age` дискретизация не задавалась, поэтому там остаются только варианты `origin` и `log`.

Полный перебор получается большим, поэтому ниже есть два режима:

- короткий запуск - быстрая проверка кода и разумный эталон;
- полный запуск - тот же код на всех комбинациях, включается отдельным флагом.

In [ ]:
transform_combinations = make_transform_combinations()
len(transform_combinations)

### 1.1 Линейная модель

Короткий запуск берёт детерминированную подвыборку комбинаций и обучает логистическую регрессию. Внутри используется многопоточность, а прогресс отображается через `tqdm`.

In [ ]:
LINEAR_TRANSFORM_SMOKE_N = 24

linear_transform_short = sample_combinations(
    transform_combinations,
    n=LINEAR_TRANSFORM_SMOKE_N,
    random_state=52,
)

linear_transform_results_short = evaluate_transform_combinations(
    data=data,
    combinations=linear_transform_short,
    model_family="linear",
    n_jobs=4,
    show_progress=True,
)

linear_transform_results_short[
    ["roc_auc", "recall", "precision", "n_features", "origin_columns", "log_columns", "discretization_columns"]
].head(5)

In [ ]:
RUN_FULL_LINEAR_TRANSFORM_SEARCH = False

if RUN_FULL_LINEAR_TRANSFORM_SEARCH:
    linear_transform_results = evaluate_transform_combinations(
        data=data,
        combinations=transform_combinations,
        model_family="linear",
        n_jobs=-1,
        show_progress=True,
    )
else:
    linear_transform_results = linear_transform_results_short

Эталонная комбинация для линейных моделей зафиксирована в проектном модуле, чтобы DVC и ноутбук использовали один и тот же выбор.

In [ ]:
selected_linear_transform_result = evaluate_transform_combinations(
    data=data,
    combinations=[SELECTED_LINEAR_TRANSFORM],
    model_family="linear",
    n_jobs=1,
    show_progress=False,
)

selected_linear_transform_result[
    ["roc_auc", "recall", "precision", "n_features", "origin_columns", "log_columns", "discretization_columns"]
]

### 1.2 Дерево решений

Для дерева используем тот же механизм перебора. На практике деревья почти не выигрывают от логарифмирования и dummy-дискретизации, но это лучше проверить тем же кодом, а не принять на веру.

In [ ]:
TREE_TRANSFORM_SMOKE_N = 48

tree_transform_short = sample_combinations(
    transform_combinations,
    n=TREE_TRANSFORM_SMOKE_N,
    random_state=53,
)

tree_transform_results_short = evaluate_transform_combinations(
    data=data,
    combinations=tree_transform_short,
    model_family="tree",
    n_jobs=4,
    show_progress=True,
)

tree_transform_results_short[
    ["roc_auc", "recall", "precision", "n_features", "origin_columns", "log_columns", "discretization_columns"]
].head(5)

In [ ]:
RUN_FULL_TREE_TRANSFORM_SEARCH = False

if RUN_FULL_TREE_TRANSFORM_SEARCH:
    tree_transform_results = evaluate_transform_combinations(
        data=data,
        combinations=transform_combinations,
        model_family="tree",
        n_jobs=-1,
        show_progress=True,
    )
else:
    tree_transform_results = tree_transform_results_short

In [ ]:
selected_tree_transform_result = evaluate_transform_combinations(
    data=data,
    combinations=[SELECTED_TREE_TRANSFORM],
    model_family="tree",
    n_jobs=1,
    show_progress=False,
)

selected_tree_transform_result[
    ["roc_auc", "recall", "precision", "n_features", "origin_columns", "log_columns", "discretization_columns"]
]

## 2. Маркеры

Теперь проверяем маркеры риска из EDA. Они уже созданы в `build_feature_space`, поэтому здесь остаётся только сформировать полный набор их комбинаций и оценить добавление маркеров поверх выбранных манипуляций с исходными признаками.

In [ ]:
marker_combinations = make_marker_combinations(include_empty=True)
len(marker_combinations)

In [ ]:
marker_summary = feature_data[MARKER_COLUMNS].agg(["sum", "mean"]).T
marker_summary["mean"] = marker_summary["mean"].map(lambda value: f"{value:.2%}")
marker_summary.sort_values("sum", ascending=False)

### 2.1 Маркеры для линейной модели

Здесь фиксируем уже выбранные манипуляции с исходными признаками и перебираем только наборы маркеров. Короткий запуск нужен для быстрой проверки; полный запуск включается тем же способом через флаг.

In [ ]:
LINEAR_MARKER_SMOKE_N = 32

linear_marker_short = sample_combinations(
    marker_combinations,
    n=LINEAR_MARKER_SMOKE_N,
    random_state=52,
)

linear_marker_results_short = evaluate_marker_combinations(
    data=data,
    transform_selection=SELECTED_LINEAR_TRANSFORM,
    marker_combinations=linear_marker_short,
    model_family="linear",
    n_jobs=4,
    show_progress=True,
)

linear_marker_results_short[
    ["roc_auc", "recall", "precision", "n_features", "marker_columns"]
].head(5)

In [ ]:
RUN_FULL_LINEAR_MARKER_SEARCH = False

if RUN_FULL_LINEAR_MARKER_SEARCH:
    linear_marker_results = evaluate_marker_combinations(
        data=data,
        transform_selection=SELECTED_LINEAR_TRANSFORM,
        marker_combinations=marker_combinations,
        model_family="linear",
        n_jobs=-1,
        show_progress=True,
    )
else:
    linear_marker_results = linear_marker_results_short

In [ ]:
selected_linear_final_result = evaluate_marker_combinations(
    data=data,
    transform_selection=SELECTED_LINEAR_TRANSFORM,
    marker_combinations=[tuple(SELECTED_LINEAR_MARKERS)],
    model_family="linear",
    n_jobs=1,
    show_progress=False,
)

selected_linear_final_result[["roc_auc", "recall", "precision", "n_features", "marker_columns"]]

### 2.2 Маркеры для дерева

Для дерева маркеры обычно дают меньше прироста, потому что дерево само строит пороговые разбиения. Но комбинированные маркеры всё равно проверяем, потому что они могут заранее подсветить редкие рискованные группы.

In [ ]:
TREE_MARKER_SMOKE_N = 64

tree_marker_short = sample_combinations(
    marker_combinations,
    n=TREE_MARKER_SMOKE_N,
    random_state=54,
)

tree_marker_results_short = evaluate_marker_combinations(
    data=data,
    transform_selection=SELECTED_TREE_TRANSFORM,
    marker_combinations=tree_marker_short,
    model_family="tree",
    n_jobs=4,
    show_progress=True,
)

tree_marker_results_short[
    ["roc_auc", "recall", "precision", "n_features", "marker_columns"]
].head(5)

In [ ]:
RUN_FULL_TREE_MARKER_SEARCH = False

if RUN_FULL_TREE_MARKER_SEARCH:
    tree_marker_results = evaluate_marker_combinations(
        data=data,
        transform_selection=SELECTED_TREE_TRANSFORM,
        marker_combinations=marker_combinations,
        model_family="tree",
        n_jobs=-1,
        show_progress=True,
    )
else:
    tree_marker_results = tree_marker_results_short

In [ ]:
selected_tree_final_result = evaluate_marker_combinations(
    data=data,
    transform_selection=SELECTED_TREE_TRANSFORM,
    marker_combinations=[tuple(SELECTED_TREE_MARKERS)],
    model_family="tree",
    n_jobs=1,
    show_progress=False,
)

selected_tree_final_result[["roc_auc", "recall", "precision", "n_features", "marker_columns"]]

## 3. Итоговые комбинации

Итог фиксируем в коде проекта, а не только в ноутбуке. Это важно: DVC-команда строит датасеты из тех же `SELECTED_*` констант, которые показаны ниже.

In [ ]:
final_choices = {
    "linear": {
        "transform": SELECTED_LINEAR_TRANSFORM,
        "markers": SELECTED_LINEAR_MARKERS,
    },
    "tree": {
        "transform": SELECTED_TREE_TRANSFORM,
        "markers": SELECTED_TREE_MARKERS,
    },
}

final_choices

## 4. DVC-датасеты

Финальные датасеты собираются DVC-этапом `build-feature-engineering-datasets`:

- `data/processed/linear/train.csv`
- `data/processed/linear/test.csv`
- `data/processed/tree/train.csv`
- `data/processed/tree/test.csv`

Команда внутри DVC вызывает проектный CLI и применяет выбранные комбинации к train и test одинаково.

In [ ]:
processed_paths = [
    PROJECT_ROOT / "data" / "processed" / "linear" / "train.csv",
    PROJECT_ROOT / "data" / "processed" / "linear" / "test.csv",
    PROJECT_ROOT / "data" / "processed" / "tree" / "train.csv",
    PROJECT_ROOT / "data" / "processed" / "tree" / "test.csv",
]

for path in processed_paths:
    if path.exists():
        print(path.relative_to(PROJECT_ROOT), pd.read_csv(path).shape)
    else:
        print(path.relative_to(PROJECT_ROOT), "not built yet")